In [ ]:
suppressWarnings(suppressMessages({
    library(limma)
    library(nlme)
    library(parallel)
    library(doParallel)
    library(foreach)
    library(EpiDISH)
    library(lumi)
    library(dplyr)
}))

# Reference generation

In [ ]:
# Run notebook with working directory set to the HOMED repository root.
lung_dir <- file.path("benchmarking", "Lung")
stopifnot(dir.exists(lung_dir))

source("R/HOMED_Reference_multilevel.r")
source("R/HOMED_Estimate_multilevel.r")
source("R/HOMED_Plot_multilevel.r")

In [ ]:
FACS_methyl <- readRDS(file.path(lung_dir, "lung_FACS_methyl.rds"))
FACS_meta <- readRDS(file.path(lung_dir, "lung_FACS_meta.rds"))
bulk_methyl_train <- readRDS(file.path(lung_dir, "lung_bulk_methyl_train.rds"))
truth <- readRDS(file.path(lung_dir, "lung_bulk_prop_DWLS_immune_4type_unstranded.rds"))

In [ ]:
# Subset truth matrix to training samples
truth_train <- truth[
    ,
    colnames(bulk_methyl_train),
    drop = FALSE
]

In [ ]:
# Keep only CpGs that are finite in every training sample
finite_train <- apply(
    bulk_methyl_train,
    1,
    function(x) all(is.finite(x))
)

cat("CpGs before filtering:", nrow(bulk_methyl_train), "\n")
cat("CpGs retained:", sum(finite_train), "\n")
cat("CpGs removed:", sum(!finite_train), "\n")

bulk_methyl_train <- bulk_methyl_train[
    finite_train,
    ,
    drop = FALSE
]

# Match FACS methylation to the cleaned training CpGs
common_cpg <- intersect(
    rownames(FACS_methyl),
    rownames(bulk_methyl_train)
)

FACS_methyl <- FACS_methyl[
    common_cpg,
    ,
    drop = FALSE
]

bulk_methyl_train <- bulk_methyl_train[
    common_cpg,
    ,
    drop = FALSE
]

In [ ]:
celltypes <- list(
    immune = NULL,
    endothelial = NULL,
    epithelial = NULL,
    fibroblast = NULL
)

In [ ]:
Lung_ref <- HOMED_Reference(
    FACS_beta = FACS_methyl, 
    FACS_pd = FACS_meta, 
    training_beta = bulk_methyl_train, 
    training_prop = t(truth_train), 
    celltypes = celltypes
)

In [ ]:
saveRDS(Lung_ref, file.path(lung_dir, "lung_ref.rds"))

# Estimation

In [ ]:
bulk_methyl_test <- readRDS(file.path(lung_dir, "lung_bulk_methyl_test.rds"))

In [ ]:
# Keep only CpGs that are finite in every testing sample
finite_test <- apply(
    bulk_methyl_test,
    1,
    function(x) all(is.finite(x))
)

cat("CpGs before filtering:", nrow(bulk_methyl_test), "\n")
cat("CpGs retained:", sum(finite_test), "\n")
cat("CpGs removed:", sum(!finite_test), "\n")

bulk_methyl_test<- bulk_methyl_test[
    finite_test,
    ,
    drop = FALSE
]

In [ ]:
# Subset truth matrix to testing samples
truth_test <- truth[, colnames(bulk_methyl_test), drop = FALSE]

In [ ]:
ref_matrix <- readRDS(file.path(lung_dir, "lung_ref.rds"))

In [ ]:
celltypes <- list(
    immune = NULL,
    endothelial = NULL,
    epithelial = NULL,
    fibroblast = NULL
)

In [ ]:
estimated_prop_test <- HOMED_Estimate(
    celltypes = celltypes,
    deconv_beta = bulk_methyl_test,
    reference_obj = ref_matrix
)$deconvolution_proportion

In [ ]:
saveRDS(estimated_prop_test, file.path(lung_dir, "est_prop.rds"))

# Produce scatterplot

In [ ]:
# ============================================================
# Prepare truth and HOMED estimates for plotting
# ============================================================

estimated_prop_test <- readRDS(file.path(lung_dir, "est_prop.rds"))
truth_test <- truth[, colnames(bulk_methyl_test), drop = FALSE]

meta_test <- t(truth_test)
est_prop <- estimated_prop_test

# Make sure samples are aligned
common_samples <- intersect(
    rownames(meta_test),
    rownames(est_prop)
)

meta_test <- meta_test[
    common_samples,
    ,
    drop = FALSE
]

est_prop <- est_prop[
    common_samples,
    ,
    drop = FALSE
]

stopifnot(
    identical(
        rownames(meta_test),
        rownames(est_prop)
    )
)


# ============================================================
# Cell type labels/order
# ============================================================

ct_names <- c(
    "epithelial"  = "Epithelial",
    "endothelial" = "Endothelial",
    "fibroblast"  = "Fibroblast",
    "immune"      = "Immune Cells"
)

ct_order <- c(
    "epithelial",
    "endothelial",
    "fibroblast",
    "immune"
)

cts <- ct_order[
    ct_order %in% colnames(meta_test) &
    ct_order %in% colnames(est_prop)
]

print(cts)


# ============================================================
# Scatterplots
# ============================================================

png(
    file.path(lung_dir, "Lung_scatter_HOMED.png"),
    width = 4000,
    height = 1000,
    res = 300
)

par(
    mfrow = c(1, 4),
    mar = c(4, 4, 2, 1),
    pty = "s"
)

# Same axes for all cell types
ylim_all <- range(
    c(
        as.matrix(meta_test[, cts, drop = FALSE]),
        as.matrix(est_prop[, cts, drop = FALSE])
    ),
    na.rm = TRUE
)

xlim_all <- ylim_all

for (ct in cts) {

    truth_ct <- meta_test[, ct]
    est_ct   <- est_prop[, ct]

    pcc <- cor(
        truth_ct,
        est_ct,
        use = "complete.obs"
    )

    rmse <- sqrt(
        mean(
            (truth_ct - est_ct)^2,
            na.rm = TRUE
        )
    )

    plot(
        truth_ct,
        est_ct,
        main = ct_names[ct],
        cex.main = 1.5,
        cex.lab = 1.5,
        font.main = 2,
        cex.axis = 1.3,
        xlab = "True",
        ylab = "Estimated",
        pch = 19,
        xlim = xlim_all,
        ylim = ylim_all,
        asp = 1
    )

    abline(
        0,
        1,
        col = "red",
        lwd = 2
    )

    legend(
        "topleft",
        legend = c(
            paste0("PCC = ", round(pcc, 2)),
            paste0("RMSE = ", round(rmse, 2))
        ),
        bty = "n",
        cex = 1.4
    )
}

dev.off()

# Simulated data

In [ ]:
sim_bulk_methyl_test <- readRDS(file.path(lung_dir, "Lung_sim_beta_sd_random.rds"))

In [ ]:
ref_matrix <- readRDS(file.path(lung_dir, "lung_ref.rds"))

In [ ]:
celltypes <- list(
    immune = NULL,
    endothelial = NULL,
    epithelial = NULL,
    fibroblast = NULL
)

In [ ]:
sim_estimated_prop_test <- HOMED_Estimate(
    celltypes = celltypes,
    deconv_beta = sim_bulk_methyl_test,
    reference_obj = ref_matrix
)$deconvolution_proportion

In [ ]:
saveRDS(sim_estimated_prop_test, file.path(lung_dir, "sim_est_prop.rds"))

In [ ]:
sim_truth <- readRDS(file.path(lung_dir, "Lung_sim_prop_sd_random.rds"))

In [ ]:
# ============================================================
# Prepare truth and HOMED estimates for plotting
# ============================================================

meta_test <- t(sim_truth)
est_prop <- sim_estimated_prop_test

colnames(meta_test) <- c(
    "immune",
    "endothelial",
    "epithelial",
    "fibroblast"
)

# Make sure samples are aligned
common_samples <- intersect(
    rownames(meta_test),
    rownames(est_prop)
)

meta_test <- meta_test[
    common_samples,
    ,
    drop = FALSE
]

est_prop <- est_prop[
    common_samples,
    ,
    drop = FALSE
]

stopifnot(
    identical(
        rownames(meta_test),
        rownames(est_prop)
    )
)


# ============================================================
# Cell type labels/order
# ============================================================

ct_names <- c(
    "epithelial"  = "Epithelial",
    "endothelial" = "Endothelial",
    "fibroblast"  = "Fibroblast",
    "immune"      = "Immune Cells"
)

ct_order <- c(
    "epithelial",
    "endothelial",
    "fibroblast",
    "immune"
)

cts <- ct_order[
    ct_order %in% colnames(meta_test) &
    ct_order %in% colnames(est_prop)
]

print(cts)


# ============================================================
# Scatterplots
# ============================================================

png(
    file.path(lung_dir, "simLung_scatter_HOMED.png"),
    width = 4000,
    height = 1000,
    res = 300
)

par(
    mfrow = c(1, length(cts)),
    mar = c(4.5, 4.5, 2.5, 1.5),
    pty = "s"
)

# Same axes for all cell types
ylim_all <- range(
    c(
        as.matrix(meta_test[, cts, drop = FALSE]),
        as.matrix(est_prop[, cts, drop = FALSE])
    ),
    na.rm = TRUE
)

xlim_all <- ylim_all

for (ct in cts) {

    truth_ct <- meta_test[, ct]
    est_ct   <- est_prop[, ct]

    pcc <- cor(
        truth_ct,
        est_ct,
        use = "complete.obs"
    )

    rmse <- sqrt(
        mean(
            (truth_ct - est_ct)^2,
            na.rm = TRUE
        )
    )

    plot(
        truth_ct,
        est_ct,
        main = ct_names[ct],
        cex.main = 1.5,
        cex.lab = 1.5,
        font.main = 2,
        cex.axis = 1.3,
        xlab = "True",
        ylab = "Estimated",
        pch = 19,
        xlim = xlim_all,
        ylim = ylim_all,
        asp = 1
    )

    abline(
        0,
        1,
        col = "red",
        lwd = 2
    )

    legend(
        "topleft",
        legend = c(
            paste0("PCC = ", round(pcc, 2)),
            paste0("RMSE = ", round(rmse, 2))
        ),
        bty = "n",
        cex = 1.4
    )
}

dev.off()